# 06 · Final test (2007–2008) - run ONCE, at the very end
Everything was chosen on 1990–2006: features, settings, calibrator, threshold, main models, error bank.
This notebook scores the **frozen** models on the test years, which no model or choice has seen.
Do not change anything and re-run to improve these numbers; if you change a model, the test is no longer fair.

**Outputs:** `outputs/results/test_*.csv` and `outputs/figures/test_*.png` - these are the numbers for the PPT.

In [ ]:
# ▶ Setup: connect Google Drive, install missing packages, load the project code
import sys, os, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "catboost", "lightgbm", "xgboost"], check=False)

PROJECT_DIR = "/content/drive/MyDrive/CycloneProject"      # <- change only if you put the project elsewhere
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

import numpy as np, pandas as pd
from src import config as C
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 60)
print("project:", C.PROJECT, "| ERA5:", C.ERA5_ROOT)

In [ ]:
import json, pickle, joblib, warnings
warnings.filterwarnings("ignore")
from sklearn.metrics import roc_auc_score
from src import occurrence as OC, metrics as M, plots as P, baselines as BL, strike as S, tracks
from src.baselines import occurrence_persistence_proba
from src.common import load_table, imd_grade
from src.labels import TARGET_TRACK, TARGET_INT
print("test years:", C.TEST)

## Occurrence

In [ ]:
FEATS = json.load(open(C.out("models") / "occurrence_features.json"))
booster = OC.load_booster(C.out("models") / "occurrence_xgb.json")
cal = pickle.load(open(C.out("models") / "occurrence_calibrator.pkl", "rb"))
THR = json.load(open(C.out("models") / "occurrence_threshold.json"))["threshold_p_cs_plus"]
KEEP = FEATS + [c for c in ["lat", "lon", "time", "year", "label", "persistence", "existing_system", "w"] if c not in FEATS]
te = OC.load_split(C.out("tables", "occurrence"), C.TEST, KEEP)
Pt = cal.predict(OC.margins(booster, te, FEATS))
Pp = occurrence_persistence_proba(te.persistence.values)
rows = {}
for g, ix in {"A: system present": te.existing_system.values == 1,
              "B: no system at t0": te.existing_system.values == 0, "all": np.ones(len(te), bool)}.items():
    y, w = te.label.values[ix], te.w.values[ix]
    bm, bp = M.brier_multi(y, Pt[ix], w), M.brier_multi(y, Pp[ix], w)
    rows[g] = dict(brier_model=bm, brier_persistence=bp, brier_skill_vs_persistence_pct=100 * (1 - bm / bp),
                   logloss_model=M.logloss_multi(y, Pt[ix], w), rows=int(ix.sum()))
occ = pd.DataFrame(rows).T; display(occ.round(4)); occ.to_csv(C.out("results") / "test_occurrence_groups.csv")

fixes = tracks.load_ibtracs(); TRK, INFO = tracks.build_tracks(fixes); TP = tracks.TrackPoints(TRK)
full = OC.load_split(C.out("tables", "occurrence_full"), C.TEST, KEEP)
full["p_cs_plus"] = cal.predict(OC.margins(booster, full, FEATS))[:, 2:].sum(axis=1)
ev = M.event_scores(full, "p_cs_plus", TP, [THR])
display(ev.round(3)); ev.to_csv(C.out("results") / "test_occurrence_events.csv", index=False)
ycs = (te.label.values >= 2).astype(int)
P.reliability_plot({"test": M.reliability(ycs, Pt[:, 2:].sum(axis=1), te.w.values)},
                   "P(CS or stronger within 200 km, 24 h) - test", C.out("figures") / "test_occurrence_reliability.png")

## Track and intensity (frozen main model vs baselines)

In [ ]:
df = load_table(C.out("tables") / "storm_fixes")
SF = json.load(open(C.out("models") / "storm_features.json"))
MAIN = json.load(open(C.out("models") / "storm_models_info.json"))["main_model"]
tr, ts = df[df.split == "train"], df[df.split == "test"].copy()
PRED = {"Persistence": pd.concat([BL.track_persistence(ts), BL.intensity_persistence(ts)], axis=1),
        "CLIPER": BL.cliper(TARGET_TRACK).fit(tr).predict(ts), "SHIFOR": BL.shifor(TARGET_INT).fit(tr).predict(ts)}
PRED[MAIN] = pd.DataFrame({t: joblib.load(C.out("models") / f"{t}.joblib").predict(ts[SF])
                           for t in TARGET_TRACK + TARGET_INT}, index=ts.index)
rows = []
for name, pr in PRED.items():
    for L in C.LEADS_H:
        if f"y_dlat_{L}" in pr:
            ok = ts[f"y_dlat_{L}"].notna().values
            rows.append(dict(model=name, lead_h=L, target="track",
                             track_error_km=np.nanmean(M.track_error_km(ts[ok], pr[ok], L)), n=int(ok.sum())))
        if f"y_dwind_{L}" in pr:
            y = ts[f"y_dwind_{L}"].values
            rows.append(dict(model=name, lead_h=L, target="intensity", mae_kt=M.mae(y, np.asarray(pr[f"y_dwind_{L}"])),
                             n=int(np.isfinite(y).sum())))
res = pd.DataFrame(rows); res.to_csv(C.out("results") / "test_track_intensity.csv", index=False)
trk, itn = res[res.target == "track"], res[res.target == "intensity"]
display(trk.pivot(index="model", columns="lead_h", values="track_error_km").round(0))
display(itn.pivot(index="model", columns="lead_h", values="mae_kt").round(1))
P.lead_lines(trk, "track_error_km", "Mean track error (km)", "Track error - TEST 2007-2008", C.out("figures") / "test_track_error.png")
P.lead_lines(itn, "mae_kt", "Mean absolute error (kt)", "Intensity error - TEST 2007-2008", C.out("figures") / "test_intensity_error.png")
print(f"test: {ts.SID.nunique()} storms, {len(ts)} storm times")

In [ ]:
# ▶ Rapid intensification and peak intensity on the test years
ok = ts.y_ri.notna().values
ri_m = joblib.load(C.out("models") / "y_ri.joblib")
y, p = ts.loc[ok, "y_ri"].values.astype(int), ri_m.predict_proba(ts.loc[ok, SF])[:, 1]
ri = pd.DataFrame([dict(model="Base rate", brier=M.brier_binary(y, np.full(len(y), tr.y_ri.mean()))),
                   dict(model=MAIN, brier=M.brier_binary(y, p),
                        roc_auc=roc_auc_score(y, p) if len(set(y)) > 1 else np.nan)])
display(ri.round(4)); ri.to_csv(C.out("results") / "test_ri.csv", index=False)
ok = ts.y_peak_wind.notna().values
pk_m = joblib.load(C.out("models") / "y_peak_wind.joblib")
yp = ts.loc[ok, "y_peak_wind"].values
pp = np.maximum(pk_m.predict(ts.loc[ok, SF]), ts.loc[ok, "wind"].values)
pk = pd.DataFrame([dict(model="Persistence", mae_kt=M.mae(yp, ts.loc[ok, "wind"].values),
                        grade_accuracy=float((imd_grade(ts.loc[ok, "wind"].values) == imd_grade(yp)).mean())),
                   dict(model=MAIN, mae_kt=M.mae(yp, pp), grade_accuracy=float((imd_grade(pp) == imd_grade(yp)).mean()))])
display(pk.round(3)); pk.to_csv(C.out("results") / "test_peak.csv", index=False)

## Coastal strike probability on the test years

In [ ]:
TM = {t: joblib.load(C.out("models") / f"{t}.joblib") for t in TARGET_TRACK}
ERR = np.load(C.out("models") / "track_error_bank.npy")
DIST, _ = S.load_districts(); rng = np.random.default_rng(C.RANDOM_SEED)
ev = ts[ts["c_dist_land_km"] < 900] if "c_dist_land_km" in ts else ts
ev = ev.iloc[::2]
fc = {t: TM[t].predict(ev[SF]) for t in TARGET_TRACK}
out = []
for i, (_, r) in enumerate(ev.iterrows()):
    hrs, HL, HN = S.simulate(r.lat, r.lon, [fc[f"y_dlat_{L}"][i] for L in C.LEADS_H],
                             [fc[f"y_dlon_{L}"][i] for L in C.LEADS_H], ERR, C.N_MC_TRACKS_VALIDATION, rng)
    out.append(S.strike_probabilities(hrs, HL, HN, DIST).merge(S.observed_hits(TRK[r.SID], r.time, DIST), on="district"))
sv = pd.concat(out, ignore_index=True)
rows = []
for L in C.LEADS_H:
    s = sv.dropna(subset=[f"o_{L}h"])
    rows.append(dict(window=f"0-{L} h", pairs=len(s), hits=int(s[f"o_{L}h"].sum()),
                     brier=M.brier_binary(s[f"o_{L}h"], s[f"p_{L}h"]),
                     brier_climatology=M.brier_binary(s[f"o_{L}h"], np.full(len(s), s[f"o_{L}h"].mean()))))
st = pd.DataFrame(rows); display(st.round(4)); st.to_csv(C.out("results") / "test_strike.csv", index=False)